In [19]:
import sklearn
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
import math
pd.__version__

'2.3.3'

In [21]:
import os
import mlflow

os.environ['AWS_PROFILE'] = 'sergio'

TRACKING_URI = '13.61.191.33'
mlflow.set_tracking_uri(f'http://{TRACKING_URI}:5000')
mlflow.set_experiment('nyc-taxi-experiment')

<Experiment: artifact_location='s3://mlflow-artifacts-remote-786875389232/1', creation_time=1794424515109, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794424515109, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [22]:
def read_data(file):
    df = pd.read_parquet(file)
    print(f"Number of columns for {file}: {len(df.columns)}")
    # print(df.dtypes)
    #convert to datetime64[ns] to int64
    converted_start_end_df = df.astype({"lpep_pickup_datetime": "int", "lpep_dropoff_datetime": "int"})
    df['duration'] = (converted_start_end_df['lpep_dropoff_datetime'] - converted_start_end_df['lpep_pickup_datetime']) * 1.6666666666667e-8
    # print(df.duration.describe())
    print(f"min: {df.duration.min()}")
    print(f"max: {df.duration.max()}")
    df_b060 = df[(df.duration >= 1) & (df.duration <= 60)]
    # Q:What fraction of the records left after you dropped the outliers?
    fraction = len(df_b060) / len(df)
    print(f"fraction: {fraction*100:.2f}")
    return df_b060[["PULocationID", "DOLocationID", "duration"]]

In [34]:
train_df = read_data("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
print("done reading data")

Number of columns for https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet: 20
min: 0.0
max: 1439.6000000000288
fraction: 96.54
done reading data


In [35]:
val_df = read_data("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")

Number of columns for https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet: 20
min: 0.0
max: 1439.6166666666957
fraction: 95.80


In [25]:
# sns.distplot(train_df.duration, label="train")
# sns.distplot(val_df.duration, label="val")
# plt.legend()

In [26]:
categorical = ["PULocationID", "DOLocationID"]
numerical = ['trip_distance']

vectorizer = DictVectorizer()

train_dicts = train_df[categorical].astype(str).to_dict(orient='records')
pprint(train_dicts[:5])
X_train = vectorizer.fit_transform(train_dicts)
del train_dicts

[{'DOLocationID': '143', 'PULocationID': '166'},
 {'DOLocationID': '43', 'PULocationID': '24'},
 {'DOLocationID': '179', 'PULocationID': '223'},
 {'DOLocationID': '238', 'PULocationID': '41'},
 {'DOLocationID': '74', 'PULocationID': '41'}]


In [27]:
val_dicts = val_df[categorical].astype(str).to_dict(orient='records')
pprint(val_dicts[:5])
X_val = vectorizer.transform(val_dicts)
del val_dicts

[{'DOLocationID': '265', 'PULocationID': '74'},
 {'DOLocationID': '196', 'PULocationID': '216'},
 {'DOLocationID': '114', 'PULocationID': '7'},
 {'DOLocationID': '239', 'PULocationID': '74'},
 {'DOLocationID': '223', 'PULocationID': '82'}]


In [28]:
print(X_val.shape)

(62571, 467)


In [29]:
target = "duration"
y_train = train_df[target].values
y_val = val_df[target].values

train = xgb.DMatrix(X_train, label=y_train)
validation = xgb.DMatrix(X_val, label=y_val)

In [30]:
del train_df
del val_df

In [31]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("developer", "sergio")
        mlflow.log_param("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=5,
            evals=[(validation, "validation")],
            early_stopping_rounds=10
        )
        y_pred = booster.predict(validation)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
    return {"loss": rmse, "status": STATUS_OK}

search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)



[0]	validation-rmse:8.85077                           
[1]	validation-rmse:8.52105                           
[2]	validation-rmse:8.25488                           
  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [12:43:40] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[3]	validation-rmse:8.05864                           
[4]	validation-rmse:7.91589                           
🏃 View run big-shad-150 at: http://13.61.191.33:5000/#/experiments/1/runs/2978443fac9b4a169766f670eb3b69dd

🧪 View experiment at: http://13.61.191.33:5000/#/experiments/1

[0]	validation-rmse:8.68148                                                    
[1]	validation-rmse:8.20990                                                    
[2]	validation-rmse:7.80145                                                    
  2%|▏         | 1/50 [00:01<00:38,  1.27trial/s, best loss: 7.915894896684681]

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [12:43:40] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[3]	validation-rmse:7.53101                                                    
[4]	validation-rmse:7.29959                                                    
🏃 View run amazing-snail-348 at: http://13.61.191.33:5000/#/experiments/1/runs/657c4a9e58bc4f8d95cd77455ffde6f8

🧪 View experiment at: http://13.61.191.33:5000/#/experiments/1                

[0]	validation-rmse:9.03367                                                    
[1]	validation-rmse:8.77227                                                    
  4%|▍         | 2/50 [00:01<00:33,  1.41trial/s, best loss: 7.299590045676925]

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [12:43:41] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



... (91 outputs left out)

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [12:44:08] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[3]	validation-rmse:6.23088                                                     
[4]	validation-rmse:6.19097                                                     
🏃 View run zealous-jay-105 at: http://13.61.191.33:5000/#/experiments/1/runs/e05356eb023c4c0fba32181e25108693

🧪 View experiment at: http://13.61.191.33:5000/#/experiments/1                 

[0]	validation-rmse:8.77736                                                     
[1]	validation-rmse:8.41078                                                     
[2]	validation-rmse:8.14288                                                     
[3]	validation-rmse:7.87372                                                     
[4]	validation-rmse:7.72558                                                     
 98%|█████████▊| 49/50 [00:29<00:00,  1.67trial/s, best loss: 6.027705541365468]

/opt/conda/envs/bootcamp/lib/python3.10/site-packages/xgboost/callback.py:385: UserWarning: [12:44:08] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



🏃 View run incongruous-wasp-925 at: http://13.61.191.33:5000/#/experiments/1/runs/0577daf848fa4297896a21b702d86084

🧪 View experiment at: http://13.61.191.33:5000/#/experiments/1                 

100%|██████████| 50/50 [00:29<00:00,  1.71trial/s, best loss: 6.027705541365468]


In [32]:
# print("Best hyperparameters:", best_result)

In [33]:
import numpy as np

with mlflow.start_run():
    mlflow.set_tag("developer", "sergio")
    mlflow.log_param("model", "xgboost")
    best_result = {'learning_rate': np.float64(0.20844413031306103), 
                   'max_depth': np.int64(99.0), 
                   'min_child_weight': np.float64(1.5432280770679585), 
                   'reg_alpha': np.float64(0.04953891913131428), 
                   'reg_lambda': np.float64(0.017060826809340544)}
    mlflow.log_params(best_result)
    booster = xgb.train(
        params=best_result,
        dtrain=train,
        num_boost_round=2,
        evals=[(validation, "validation")],
        early_stopping_rounds=10
    )
    y_pred = booster.predict(validation)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
    mlflow.xgboost.log_model(booster, name="models_mlflow")

[0]	validation-rmse:8.28000
[1]	validation-rmse:7.56324
🏃 View run secretive-vole-744 at: http://13.61.191.33:5000/#/experiments/1/runs/cb6656d50a87462aa4c8eea53731c0b0
🧪 View experiment at: http://13.61.191.33:5000/#/experiments/1
